# AIHub CallcenterQA_Audio (고객응대 음성) — EDA

`/data/ASR/RAW/AIHub_CallcenterQA_Audio`

## 목적
- 구조/인벤토리 → 라벨 스키마 → 오디오 규격 → 파서 → 전체 EDA → SILVER(transcript.jsonl)

## 예상 특이사항
- 콜센터/고객응대 계열 → **8kHz 전화망 가능성** (단 WelfareCounsel처럼 16k 스마트폰 녹음일 수도 — 확인 필수)
- 기존 고객응대 데이터(011)와 같은 `dataSet.dialogs` 스키마일 가능성 — 그러면 CounselingSpeech 파서 재사용
- 지금까지 본 스키마 3형: ①CounselingSpeech형(세션 JSON+발화 txt) ②LowQuality형(세션 JSON에 text 포함) ③WelfareCounsel형(발화 단위 JSON)

## 접근
탐색 셀 1~4 출력 확인 → 파서·전체 EDA → 신규 포맷 빌드

In [1]:
from pathlib import Path
import re, json, random, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR 확인")

ROOT = Path("/data/ASR/RAW/AIHub_CallcenterQA_Audio")
AUDIO_EXT = {".wav", ".flac", ".pcm", ".mp3", ".m4a"}

def detect_split(p):
    s = str(p).lower()
    return "train" if "train" in s else "valid" if "valid" in s else "test" if "test" in s else "unknown"

def tree(path, prefix="", depth=3, maxn=8):
    try:
        dirs = [p for p in sorted(Path(path).iterdir()) if p.is_dir()]
    except Exception:
        return
    for p in dirs[:maxn]:
        print(prefix + p.name + "/")
        if depth > 0:
            tree(p, prefix + "  ", depth - 1, maxn)
    if len(dirs) > maxn:
        print(prefix + f"... (+{len(dirs)-maxn} dirs)")

def walk_keys(obj, prefix="", out=None, depth=8):
    if out is None: out = []
    if depth < 0:   return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            walk_keys(v, f"{prefix}.{k}" if prefix else k, out, depth-1)
    elif isinstance(obj, list):
        out.append((prefix + "[]", f"list(len={len(obj)})", ""))
        if obj: walk_keys(obj[0], prefix + "[0]", out, depth-1)
    else:
        out.append((prefix, type(obj).__name__, str(obj)[:60]))
    return out

print("ROOT 존재:", ROOT.is_dir())

ROOT 존재: True


## 1. 디렉터리 구조 / 인벤토리

In [2]:
print("=== 디렉터리 구조 (상위 3단계) ===")
tree(ROOT, depth=3)

json_files = list(ROOT.rglob("*.json"))
txt_files  = list(ROOT.rglob("*.txt"))
audio = []
for ext in AUDIO_EXT:
    audio += list(ROOT.rglob(f"*{ext}"))
zips = list(ROOT.rglob("*.zip"))

print(f"\nJSON {len(json_files):,} | TXT {len(txt_files):,} | 오디오 {len(audio):,} | zip {len(zips):,}")
if zips:
    print("⚠ zip 목록(백업/미해제 확인):", [z.name for z in zips[:8]])
print("split 분포(json) :", dict(Counter(detect_split(p) for p in json_files)))
print("split 분포(오디오):", dict(Counter(detect_split(p) for p in audio)))
print("오디오 확장자    :", dict(Counter(p.suffix.lower() for p in audio)))

=== 디렉터리 구조 (상위 3단계) ===
022.민원(콜센터)_질의-응답_데이터/
  01.데이터/
    1.Training/
      라벨링데이터_220121_add/
      라벨링데이터_220125_add/
      라벨링데이터_231222_add/
      원천데이터_220125_add/
      원천데이터_220325_add/
    2.Validation/
      라벨링데이터_220121_add/
      라벨링데이터_220125_add/
      라벨링데이터_231222_add/
      원천데이터_220125_add/
      원천데이터_220325_add/

JSON 50 | TXT 4 | 오디오 10,437 | zip 100
⚠ zip 목록(백업/미해제 확인): ['AS.zip', '결제.zip', '반품.zip', '교환.zip', '주문.zip', '업무처리.zip', '배송.zip', '증상징후1.zip']
split 분포(json) : {'train': 25, 'valid': 25}
split 분포(오디오): {'train': 9223, 'valid': 1214}
오디오 확장자    : {'.m4a': 4312, '.mp3': 6125}


## 2. 라벨 JSON 스키마 (서로 다른 위치 2개 비교)

In [3]:
if json_files:
    for sj in [json_files[0], json_files[len(json_files)//2]][:2]:
        print("=" * 60)
        print("검사 파일:", sj.relative_to(ROOT))
        try:
            data = json.loads(sj.read_text(encoding="utf-8", errors="replace"))
        except Exception as e:
            print("  파싱 실패:", e); continue
        for path, typ, ex in walk_keys(data):
            print(f"  {path:48s} {typ:14s} {ex}")
else:
    print("JSON 없음")

검사 파일: 022.민원(콜센터)_질의-응답_데이터/01.데이터/1.Training/라벨링데이터_220121_add/다산콜센터/민원(콜센터)_질의응답_다산콜센터_코로나19_관련_상담_Training/민원(콜센터) 질의응답_다산콜센터_코로나19 관련 상담_Training.json
  []                                               list(len=32475) 
  [0].도메인                                          str            다산콜센터
  [0].카테고리                                         str            코로나19 관련 상담
  [0].대화셋일련번호                                      str            B2425
  [0].화자                                           str            고객
  [0].문장번호                                         str            1
  [0].고객의도                                         str            정부지원
  [0].상담사의도                                        str            
  [0].QA                                           str            Q
  [0].고객질문(요청)                                     str            자가격리시 정부의 지원에 대해 알고싶습니다.
  [0].상담사질문(요청)                                    str            
  [0].고객답변                                         st

## 3. 전사(.txt) 형식·인코딩 (있다면)

In [4]:
if txt_files:
    for t in txt_files[:3]:
        raw = t.read_bytes()
        enc_ok, s = None, ""
        for enc in ("utf-8", "cp949", "euc-kr"):
            try:
                s = raw.decode(enc); enc_ok = enc; break
            except Exception:
                continue
        print(f"{t.relative_to(ROOT)}  [{enc_ok}]")
        print("  내용:", repr(s[:150]))
else:
    print("txt 없음 — 전사가 JSON 안에 있을 가능성")

022.민원(콜센터)_질의-응답_데이터/01.데이터/1.Training/라벨링데이터_220125_add/쇼핑/readme_K쇼핑_Training.txt  [cp949]
  내용: '\r\n민원(콜센터) 질의응답_K쇼핑_AS_Training - 약 6,940 대화쌍\r\n\r\n민원(콜센터) 질의응답_K쇼핑_결제_Training - 약 106,910 대화쌍\r\n\r\n민원(콜센터) 질의응답_K쇼핑_교환_Training - 약 50,360 대화쌍\r\n\r\n민원(콜센터)'
022.민원(콜센터)_질의-응답_데이터/01.데이터/1.Training/라벨링데이터_220125_add/질병관리본부/readme_질병관리본부_Training.txt  [cp949]
  내용: '\r\n민원(콜센터) 질의응답_질병관리본부_건강질병1_Training - 약 31,820 대화쌍\r\n\r\n민원(콜센터) 질의응답_질병관리본부_약품식품1_Training - 약 10,110 대화쌍\r\n\r\n민원(콜센터) 질의응답_질병관리본부_온라인신고_Training - 약 51,'
022.민원(콜센터)_질의-응답_데이터/01.데이터/2.Validation/라벨링데이터_220125_add/쇼핑/readme_K쇼핑_Validation.txt  [cp949]
  내용: '\r\n민원(콜센터) 질의응답_K쇼핑_AS_Validation - 약 870 대화쌍\r\n\r\n민원(콜센터) 질의응답_K쇼핑_결제_Validation - 약 13,370 대화쌍\r\n\r\n민원(콜센터) 질의응답_K쇼핑_교환_Validation - 약 6,290 대화쌍\r\n\r\n민원(콜센'


## 4. 오디오 규격 — sr(8k? 16k?)/채널/비트

In [5]:
if audio:
    samp = random.Random(0).sample(audio, min(80, len(audio)))
    rows = []
    for p in samp:
        try:
            if p.suffix.lower() == ".pcm":
                rows.append(("PCM(헤더없음)", "?", "?", None)); continue
            i = sf.info(str(p))
            rows.append((i.samplerate, i.channels, i.subtype, round(i.frames/i.samplerate, 2)))
        except Exception as e:
            rows.append(("ERR", str(e)[:25], "", None))
    a = pd.DataFrame(rows, columns=["sr", "ch", "subtype", "dur"])
    print("표본", len(a), "개")
    print("sample_rate :", a["sr"].value_counts().to_dict())
    print("channels    :", a["ch"].value_counts().to_dict())
    print("subtype     :", a["subtype"].value_counts().to_dict())
    if len(a["dur"].dropna()):
        print(f"길이(초): 평균 {a['dur'].dropna().mean():.2f} / 중앙 {a['dur'].dropna().median():.2f} / 최대 {a['dur'].dropna().max():.2f}")
else:
    print("오디오 없음")

표본 80 개
sample_rate : {48000: 36, 'ERR': 29, 44100: 15}
channels    : {2: 48, "Error opening '/data/ASR/": 29, 1: 3}
subtype     : {'MPEG_LAYER_III': 51, '': 29}
길이(초): 평균 81.26 / 중앙 83.14 / 최대 153.02
